# PATH MANAGEMENT

In [39]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

%load_ext autoreload
%autoreload 2
# %matplotlib inline

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/app
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [40]:
from src.config import Configuration

CONFIG = Configuration(
    model_name="meta-llama/Llama-2-7b-hf",

    batch_size=8,
    max_tok_length=16,

    max_epoch=5
)

# Fine-tuning

Fine-tuning refers to the process in transfer learning in which the parameter values of a model trained on a large dataset are modified when the training process continues on a small dataset (see [Kevin Murphy's book](https://probml.github.io/pml-book/book1.html) Section 19.2 for further details). The main motivation is to adapt a pre-trained model trained on a large amount of data to tackle a specific task providing better performance that would be achieved training on the small task-specific dataset.

In this notebook, we are going to use for fine-tuning a dataset set that is already available in the [Datasets repository](https://huggingface.co/datasets) from Hugging Face. However, the [Datasets library](https://huggingface.co/docs/datasets) makes easy to access and load datasets. For example, you can easily load your own dataset following [this tutorial](https://huggingface.co/docs/datasets/loading#local-and-remote-files).

More precisely, we are going to explain how to fine-tune the [Llama2 model](https://huggingface.co/docs/transformers/model_doc/llama2) on the [Europarl-ST dataset](https://huggingface.co/datasets/tj-solergibert/Europarl-ST), but only that [dataset of Europarl-ST focused on the text data for MT from English](https://huggingface.co/datasets/tj-solergibert/Europarl-ST-processed-mt-en).

In [41]:
# from datasets import load_dataset

# raw_datasets = load_dataset("tj-solergibert/Europarl-ST-processed-mt-en")

# print(raw_datasets)

from src.data import get_es_eo_dataset

raw_datasets = get_es_eo_dataset(CONFIG)

print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 200965
    })
    test: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 43063
    })
    valid: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 43063
    })
})


As shown, the Europarl-ST already comes with a pre-defined partition on the three conventional sets: training, validation and test. Each set is a dictionary with a list of source sentences (source_text), target sentences (dest_text) and the target language (dest_lang).

Let's take a closer look at the features of the training set:

In [42]:
raw_datasets["train"].features

{'source_text': Value('string'),
 'dest_text': Value('string'),
 'dest_lang': Value('int64')}

As you can see, the possible target languages are German, English, Spanish, French, Italian, Dutch, Polish, Portuguese and Romanian.

Let us take a look at the translations of the first two English sentences:

In [43]:
raw_datasets["train"][:14]["source_text"]

['artículo anteriorse devela un misterio: ¿por qué moni argento era de tostado?',
 'en el siglo iii surgieron un número de tribus germánicas del oeste grandes: alemanni, francos, catos, \ufeffsajones, frisii, \ufeffsicambri, y thuringii\ufeff.',
 'hubo unos 200 invitados.',
 '¿eres tú mayor de edad?',
 '-"pero no tienes dinero, ¿verdad?"',
 'así que, cuando ella te deja, ¿de donde crees que ella va a hacer a continuación.',
 'para empezar, como ya hemos dicho, debemos tomar la fruta con el estómago vacío.',
 'soy una viuda con cuatro hijos y me quedé atrapado en una situación financiera desde abril de 2016 y necesitaba refinanciar y pagar mis cuentas.',
 'el trabajo con espacios en blanco debe comenzar a fines de la primavera o principios del verano y no retrasarse hasta el otoño para evitar problemas e interrupciones.',
 'es la riqueza guardada por su dueño para su propia desgracia.',
 'buscamos una canción que trate sobre alguno de los siguientes temas: «desarrollo global» o «un solo

In [44]:
raw_datasets["train"][:14]["dest_text"]

['estis mistero por la polico : kial ŝteli nur unu ŝuon anstataù paro ?',
 'la tria jarcento vidis la aperon de kelkaj grandaj okcident ĝermanaj triboj: la alemanoj, frankoj, bavarii-, ĥatoj, saksoj, frisii, sicambri, kaj thuringii.',
 'venis ĉirkaŭ 200 gastoj.',
 'ĉu vi estas la plej aĝa?',
 '"sed vi ne posedas tiom da mono, ĉu ne?"',
 'do, kiam ŝi lasas vin, kie vi kredas, ke ŝi faros poste.',
 'kiel antaŭe menciite, la drogo devas esti prenita sur malplena stomako.',
 'en ĉi tiu tempo mi estas vidvino kun kvar infanoj kaj mi estis ligita en financa situacio en majo 2018 kaj bezonis refinanci kaj pagi miajn biletojn.',
 'laboro kun spacoj devas komenciĝi fine de printempo aŭ frua somero kaj ne malhelpu ĝis aŭtuno por eviti problemojn kaj interrompojn.',
 'riĉecon konservatan por la malutilo de ĝia propra mastro.',
 'tie ĉi mi menciu nur unu temaron, tiun de tutmondiĝo aŭ „globaliĝo”.',
 'ni serĉu rekte la titolon «orientaj tapiŝoj»!',
 'tamen, ĉu tranĉeoj estas por ke ni koncentriĝu'

In [45]:
raw_datasets["train"][:14]["dest_lang"]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

As shown, each English sentence is repeated for each of the seven target languages (0: 'de', 2: 'es', 3: 'fr', 4: 'it', 5: 'nl', 6: 'pl', 7: 'pt').

The Llama2 model is a pretrained Large Language Model (LLM) ready to tackle several NLP tasks, being one of the them the translation from English into Spanish. Let us filter the Europarl-ST only for English into Spanish using a simple [lambda function](https://realpython.com/python-lambda/) with the [Dataset.filter() function](https://huggingface.co/docs/datasets/v2.9.0/en/package_reference/main_classes#datasets.Dataset.filter).

In [46]:
# lang="es"
# lang_id = raw_datasets["train"].features["dest_lang"].names.index(lang)
# raw_datasets = raw_datasets.filter(lambda x: x["dest_lang"] == lang_id)

More precisely, we are going to be using the Llama-2 checkpoint [meta-llama/Llama-2-7b-hf](https://huggingface.co/meta-llama/Llama-2-7b-hf) to run our experiments for which you need to accept the LLAMA 2 COMMUNITY LICENSE AGREEMENT. Processing your request may take some time, so please do it in advance.

Logging in HuggingFace to be granted access to Llama2 with 7B parameters:

In [47]:
import os
import dotenv
from huggingface_hub import login

dotenv.load_dotenv()
login(token=os.getenv("HUGGING_FACE_TOKEN"))

python-dotenv could not parse statement starting at line 2
python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 8
python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 8


We can apply the tokenizer function to any dataset taking advantage that Hugging Face Datasets are [Apache Arrow](https://arrow.apache.org) files stored on the disk, so you only keep the samples you ask for loaded in memory.

To keep the data as a dataset, we will use the [Dataset.map() function](https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset.map). This also allows us some extra flexibility, if we need more preprocessing done than just tokenization. The map() method works by applying a function on each element of the dataset.

In our case, each sample pair is going to be preprocessed according to the needs of the model that is to be fine-tuned. In the case of Llama2, it is recommended to explicitly state a task prompt for each source sentence:

In [48]:
from transformers import AutoTokenizer

checkpoint = CONFIG.model_name
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint, use_auth_token=True,
    padding=True,
    pad_to_multiple_of=8,
    truncation=True,
    max_length=CONFIG.max_tok_length,
    padding_side='left',
    )
tokenizer.pad_token = tokenizer.eos_token

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/transformers/models/auto/tokenization_auto.py:1025: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


In [49]:
def preprocess_function(sample):
    model_inputs = tokenizer(
        sample["source_text"], 
        text_target = sample["dest_text"],
        )
    return model_inputs

The way the Datasets library applies this processing is by adding new fields to the datasets, one for each key in the dictionary returned by the tokenize function, that is, *input_ids*, *attention_mask* and *labels*. We can check what the preprocess_function is doing with a small sample

In [50]:
sample = raw_datasets["train"].select(range(2))
model_input = preprocess_function({
    "source_text": list(sample["source_text"]),
    "dest_text": list(sample["dest_text"]),
})
print(model_input)

{'input_ids': [[1, 1616, 21825, 14123, 344, 316, 955, 29874, 443, 286, 1531, 601, 29901, 18613, 1971, 439, 29948, 1601, 29875, 1852, 9239, 3152, 316, 304, 303, 912, 29973], [1, 427, 560, 14521, 474, 2236, 25300, 10243, 443, 13831, 316, 9434, 375, 22593, 1715, 5070, 628, 288, 4196, 13830, 29901, 20712, 9889, 29892, 2524, 3944, 29892, 274, 4507, 29892, 29871, 30143, 29879, 1175, 2873, 29892, 1424, 275, 2236, 29892, 29871, 30143, 29879, 293, 1117, 374, 29892, 343, 266, 3864, 2236, 30143, 29889]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'labels': [[1, 707, 275, 286, 1531, 29877, 1277, 425, 1248, 1417, 584, 413, 616, 29871, 31805, 29873, 5037, 5595, 443, 29884, 29871, 31805, 29884, 265, 385, 303, 532, 30071, 610, 29877, 1577], [1, 425, 260, 2849, 14631, 1760, 29877, 7840

In [51]:
for sample in model_input['input_ids']:
    print(tokenizer.convert_ids_to_tokens(sample))

['<s>', '▁art', 'ículo', '▁anterior', 'se', '▁de', 'vel', 'a', '▁un', '▁m', 'ister', 'io', ':', '▁¿', 'por', '▁qu', 'é', '▁mon', 'i', '▁arg', 'ento', '▁era', '▁de', '▁to', 'st', 'ado', '?']
['<s>', '▁en', '▁el', '▁siglo', '▁i', 'ii', '▁surg', 'ieron', '▁un', '▁número', '▁de', '▁trib', 'us', '▁germ', 'án', 'icas', '▁del', '▁o', 'este', '▁grandes', ':', '▁alem', 'anni', ',', '▁fran', 'cos', ',', '▁c', 'atos', ',', '▁', '\ufeff', 's', 'aj', 'ones', ',', '▁fr', 'is', 'ii', ',', '▁', '\ufeff', 's', 'ic', 'amb', 'ri', ',', '▁y', '▁th', 'uring', 'ii', '\ufeff', '.']


We can recover the source text by applying [batch_decode](https://huggingface.co/docs/transformers/en/internal/tokenization_utils#transformers.PreTrainedTokenizerBase.batch_decode) of the tokenizer 

In [52]:
tokenizer.batch_decode(model_input['input_ids'])

['<s> artículo anteriorse devela un misterio: ¿por qué moni argento era de tostado?',
 '<s> en el siglo iii surgieron un número de tribus germánicas del oeste grandes: alemanni, francos, catos, \ufeffsajones, frisii, \ufeffsicambri, y thuringii\ufeff.']

Now, we can apply the preprocess_function to the raw datasets (training, validation and test):

In [53]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/200965 [00:00<?, ? examples/s]

Map: 100%|██████████| 43063/43063 [00:00<00:00, 51630.18 examples/s]


We are going to filter the tokenized datasets by maximum number of tokens in source and target language:

In [54]:
tokenized_datasets = tokenized_datasets.filter(lambda x: len(x["input_ids"]) <= CONFIG.max_tok_length and len(x["labels"]) <= CONFIG.max_tok_length , desc=f"Discarding source and target sentences with more than {CONFIG.max_tok_length} tokens")

Discarding source and target sentences with more than 16 tokens:   0%|          | 0/200965 [00:00<?, ? examples/s]

Discarding source and target sentences with more than 16 tokens: 100%|██████████| 200965/200965 [00:03<00:00, 62156.63 examples/s]
Discarding source and target sentences with more than 16 tokens: 100%|██████████| 200965/200965 [00:03<00:00, 62156.63 examples/s]
Discarding source and target sentences with more than 16 tokens: 100%|██████████| 43063/43063 [00:00<00:00, 50608.19 examples/s]
Discarding source and target sentences with more than 16 tokens: 100%|██████████| 43063/43063 [00:00<00:00, 50608.19 examples/s]
Discarding source and target sentences with more than 16 tokens: 100%|██████████| 43063/43063 [00:00<00:00, 66242.81 examples/s]
Discarding source and target sentences with more than 16 tokens: 100%|██████████| 43063/43063 [00:00<00:00, 66242.81 examples/s]


We can take a quick look at the length histogram in the source language:

In [55]:
dic = {}
for sample in tokenized_datasets['train']:
    sample_length = len(sample['input_ids'])
    if sample_length not in dic:
        dic[sample_length] = 1
    else:
        dic[sample_length] += 1 

for i in range(1,CONFIG.max_tok_length+1):
    if i in dic:
        print(f"{i:>2} {dic[i]:>3}")

 3  22
 4 234
 5 1075
 6 2905
 7 5647
 8 8276
 9 9821
10 9751
11 8611
12 7037
13 5188
14 3638
15 2349
16 1486


Checking a sample after filtering by maximum number of tokens:

In [56]:
for sample in tokenized_datasets['train'].select(range(5)):
    print(sample['input_ids'])
    print(sample['attention_mask'])
    print(sample['labels'])

[1, 19766, 29877, 22660, 29871, 29906, 29900, 29900, 2437, 277, 2255, 29889]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 6003, 275, 29871, 31431, 381, 1335, 30520, 29871, 29906, 29900, 29900, 10489, 517, 29926, 29889]
[1, 18613, 11175, 260, 30030, 9105, 316, 1226, 328, 29973]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 29871, 31431, 29884, 3516, 22388, 425, 5644, 29926, 263, 31303, 29874, 29973]
[1, 3133, 29877, 27044, 912, 1919, 25370, 1941]
[1, 1, 1, 1, 1, 1, 1, 1]
[1, 1146, 30520, 336, 5748, 29892, 3006, 912]
[1, 337, 1789, 316, 3966, 10183, 313, 29882, 5427, 29871, 29896, 29955, 29900, 29955, 29897]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 2614, 417, 10112, 601, 313, 31303, 275, 29871, 29896, 29955, 29900, 29955, 29897]
[1, 1346, 29894, 14054, 17926, 443, 553, 579, 276, 5264, 8643]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 6836, 22388, 6427, 440, 609, 1175, 5374, 273, 29466, 23364, 18753, 29889, 2047]


In [57]:
import torch

src = CONFIG.src_abr
tgt = CONFIG.tgt_abr
task_prefix = f"Translate from {src} to {tgt}:\n"
s = ""

prefix_tok_len = len(tokenizer.encode(f"{task_prefix}{src}: {s} = {tgt}: "))
max_tok_len = prefix_tok_len
# Adding 2 for new line in target sentence and eos_token_id token
max_tok_len += 2 * CONFIG.max_tok_length + 2


def preprocess4training_function(sample):
    
    sample_size = len(sample["source_text"])

    # Creating the prompt with the task description for each source sentence
    inputs  = [f"{task_prefix}{src}: {s} = {tgt}: " for s in sample["source_text"]]

    # Appending new line after each sample in the batch
    targets = [f"{s}\n" for s in sample["dest_text"]]

    # Applying the Llama2 tokenizer to the inputs and targets 
    # to obtain "input_ids" (token_ids) and "attention mask" 
    model_inputs = tokenizer(inputs)
    labels = tokenizer(targets)
    
    # Each input is appended with its target 
    # Each target is prepended with as many special token id (-100) as the original input length
    # Both input and target (label) has the same max_tok_len
    # Attention mask is all 1s 
    for i in range(sample_size):
        sample_input_ids = model_inputs["input_ids"][i]
        label_input_ids = labels["input_ids"][i] + [tokenizer.eos_token_id]
        model_inputs["input_ids"][i] = sample_input_ids + label_input_ids
        labels["input_ids"][i] = [-100] * len(sample_input_ids) + label_input_ids
        model_inputs["attention_mask"][i] = [1] * len(model_inputs["input_ids"][i])

    # Each input is applied left padding up to max_tok_len
    # Attention mask is 0 for padding
    # Each target (label) is left filled with special token id (-100)
    # Finally inputs, attention_mask and targets (labels) are truncated to max_tok_len
    for i in range(sample_size):
        sample_input_ids = model_inputs["input_ids"][i]
        label_input_ids = labels["input_ids"][i]
        model_inputs["input_ids"][i] = [tokenizer.pad_token_id] * (
            max_tok_len - len(sample_input_ids)
        ) + sample_input_ids
        model_inputs["attention_mask"][i] = [0] * (max_tok_len - len(sample_input_ids)) + model_inputs[
            "attention_mask"
        ][i]
        labels["input_ids"][i] = [-100] * (max_tok_len - len(sample_input_ids)) + label_input_ids
        model_inputs["input_ids"][i] = torch.tensor(model_inputs["input_ids"][i][:max_tok_len])
        model_inputs["attention_mask"][i] = torch.tensor(model_inputs["attention_mask"][i][:max_tok_len])
        labels["input_ids"][i] = torch.tensor(labels["input_ids"][i][:max_tok_len])
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


We can check what the preprocess4training_function is doing:

In [58]:
sample = tokenized_datasets['train'].select(range(2))
model_input = preprocess4training_function(sample)
print(model_input)
print(tokenizer.batch_decode(model_input.input_ids))

{'input_ids': [tensor([    2,     2,     2,     2,     2,     2,     1,  4103,  9632,   515,
          831,   304,   321, 29877, 29901,    13,   267, 29901, 19766, 29877,
        22660, 29871, 29906, 29900, 29900,  2437,   277,  2255, 29889,   353,
          321, 29877, 29901, 29871,     1,  6003,   275, 29871, 31431,   381,
         1335, 30520, 29871, 29906, 29900, 29900, 10489,   517, 29926, 29889,
           13,     2]), tensor([    2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
            2,     1,  4103,  9632,   515,   831,   304,   321, 29877, 29901,
           13,   267, 29901, 18613, 11175,   260, 30030,  9105,   316,  1226,
          328, 29973,   353,   321, 29877, 29901, 29871,     1, 29871, 31431,
        29884,  3516, 22388,   425,  5644, 29926,   263, 31303, 29874, 29973,
           13,     2])], 'attention_mask': [tensor([0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

We need to replace -100 by 0 to apply batch_decode:

In [59]:
import numpy as np
for i in range(len(model_input['labels'])):
  print(tokenizer.batch_decode([np.where(model_input['labels'][i] < 0, tokenizer.pad_token_id, model_input['labels'][i])]))

['</s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s><s> venis ĉirkaŭ 200 gastoj.\n</s>']
['</s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s><s> ĉu vi estas la plej aĝa?\n</s>']


In the case of the test set, we just preprocess the inputs (source sentences)

In [60]:
def preprocess4test_function(sample):
    inputs = [f"{task_prefix}{src}: {s} = {tgt}: " for s in sample["source_text"]]
    model_inputs = tokenizer(inputs,padding=True,)
    return model_inputs

We can check what the preprocess4test_function is doing:

In [61]:
sample = tokenized_datasets['train'].select(range(2))
model_input = preprocess4test_function(sample)
print(model_input)
print(tokenizer.batch_decode(model_input.input_ids))

{'input_ids': [[1, 4103, 9632, 515, 831, 304, 321, 29877, 29901, 13, 267, 29901, 19766, 29877, 22660, 29871, 29906, 29900, 29900, 2437, 277, 2255, 29889, 353, 321, 29877, 29901, 29871], [2, 2, 1, 4103, 9632, 515, 831, 304, 321, 29877, 29901, 13, 267, 29901, 18613, 11175, 260, 30030, 9105, 316, 1226, 328, 29973, 353, 321, 29877, 29901, 29871]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}
['<s> Translate from es to eo:\nes: hubo unos 200 invitados. = eo: ', '</s></s><s> Translate from es to eo:\nes: ¿eres tú mayor de edad? = eo: ']


Preprocessing train and dev sets:

In [62]:
preprocessed_train_dataset = tokenized_datasets['train'].map(preprocess4training_function, batched=True)
preprocessed_dev_dataset = tokenized_datasets['valid'].map(preprocess4training_function, batched=True)

Map:   0%|          | 0/66040 [00:00<?, ? examples/s]

Map: 100%|██████████| 14179/14179 [00:00<00:00, 22388.02 examples/s]


In [63]:
for sample in preprocessed_train_dataset.select(range(5)):
    print(sample['input_ids'])
    print(sample['attention_mask'])
    print(sample['labels'])

[2, 2, 2, 2, 2, 2, 1, 4103, 9632, 515, 831, 304, 321, 29877, 29901, 13, 267, 29901, 19766, 29877, 22660, 29871, 29906, 29900, 29900, 2437, 277, 2255, 29889, 353, 321, 29877, 29901, 29871, 1, 6003, 275, 29871, 31431, 381, 1335, 30520, 29871, 29906, 29900, 29900, 10489, 517, 29926, 29889, 13, 2]
[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 1, 6003, 275, 29871, 31431, 381, 1335, 30520, 29871, 29906, 29900, 29900, 10489, 517, 29926, 29889, 13, 2]
[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 4103, 9632, 515, 831, 304, 321, 29877, 29901, 13, 267, 29901, 18613, 11175, 260, 30030, 9105, 316, 1226, 328, 29973, 353, 321, 29877, 29901, 29871, 1, 29871, 31431, 29884, 3516, 22388, 425, 5644, 299

Preprocessing test set:

In [64]:
preprocessed_test_dataset = tokenized_datasets['test'].map(preprocess4test_function, batched=True)

Map:   0%|          | 0/14342 [00:00<?, ? examples/s]

Map: 100%|██████████| 14342/14342 [00:00<00:00, 31166.45 examples/s]


In [65]:
for sample in preprocessed_test_dataset.select(range(5)):
    print(sample['input_ids'])
    print(sample['attention_mask'])
    print(sample['labels'])

[2, 2, 2, 1, 4103, 9632, 515, 831, 304, 321, 29877, 29901, 13, 267, 29901, 633, 307, 1232, 288, 14736, 29892, 553, 29880, 398, 1182, 912, 29889, 353, 321, 29877, 29901, 29871]
[0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 3737, 4439, 571, 8247, 3431, 352, 3848, 29876, 29892, 413, 2156, 595, 3922, 29889]
[2, 2, 2, 2, 2, 2, 1, 4103, 9632, 515, 831, 304, 321, 29877, 29901, 13, 267, 29901, 21135, 5821, 381, 1715, 425, 907, 655, 1290, 29889, 353, 321, 29877, 29901, 29871]
[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 260, 698, 29899, 29872, 2212, 29926, 3516, 5821, 294, 413, 5803, 29889]
[2, 2, 2, 1, 4103, 9632, 515, 831, 304, 321, 29877, 29901, 13, 267, 29901, 560, 3348, 11088, 316, 12374, 1647, 831, 19615, 417, 1702, 26161, 29889, 353, 321, 29877, 29901, 29871]
[0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 13457, 537, 4402, 22

[bitsandbytes](https://huggingface.co/docs/bitsandbytes/main/en/index) is a quantization library with a Transformers integration. With this integration, you can quantize a model to 8 or 4-bits and enable many other options by configuring the BitsAndBytesConfig class. For example, you can:

<ul>
<li>set load_in_4bit=True to quantize the model to 4-bits when you load it</li>
<li>set bnb_4bit_quant_type="nf4" to use a special 4-bit data type for weights initialized from a normal distribution</li>
<li>set bnb_4bit_use_double_quant=True to use a nested quantization scheme to quantize the already quantized weights</li>
<li>set bnb_4bit_compute_dtype=torch.bfloat16 to use bfloat16 for faster computation</li>
</ul>


In [66]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

Pass the quantization_config to the from_pretrained method.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    token=True,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
    
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.23s/it]



Next, you should call the prepare_model_for_kbit_training() function to preprocess the quantized model for training.

In [68]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False, gradient_checkpointing_kwargs={'use_reentrant':False})

[LoRA (Low-Rank Adaptation of Large Language Models)](https://huggingface.co/docs/peft/task_guides/lora_based_methods) is a [parameter-efficient fine-tuning (PEFT)](https://huggingface.co/docs/peft/index) technique that significantly reduces the number of trainable parameters. It works by inserting a smaller number of new weights into the model and only these are trained. This makes training with LoRA much faster, memory-efficient, and produces smaller model weights (a few hundred MBs), which are easier to store and share.

Each PEFT method is defined by a PeftConfig class that stores all the important parameters for building a PeftModel. For example, to train with LoRA, load and create a LoraConfig class and specify the following parameters:

<ul>
<li>task_type: the task to train for (sequence-to-sequence language modeling in this case)</li>
<li>r: the dimension of the low-rank matrices</li>
<li>lora_alpha: the scaling factor for the low-rank matrices</li>
<li>target_modules: determine what set of parameters are adapted</li>
<li>lora_dropout: the dropout probability of the LoRA layers</li>
</ul>

In [69]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    inference_mode=False,
)

Once LoRA and the quantization are setup, create a quantized PeftModel with the get_peft_model() function. It takes a quantized model and the LoraConfig containing the parameters for how to configure a model for training with LoRA.

In [70]:
lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()

trainable params: 8,388,608 || all params: 6,746,804,224 || trainable%: 0.1243


The function that is responsible for putting together samples inside a batch is called a collate function.

In [71]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)

## Training

The first step before we can define our [Trainer](https://huggingface.co/docs/transformers/en/main_classes/trainer) is to define a [TrainingArguments class](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.TrainingArguments) that will contain all the hyperparameters the Trainer will use for training and evaluation. The only compulsory argument you have to provide is a directory where the trained model will be saved, as well as the checkpoints along the way. For all the rest, you can set them depending on the recommendations from the model developers:

In [34]:
from transformers import TrainingArguments

gradient_accumulation_steps = 8
args = TrainingArguments(
    CONFIG.output_model_dir,
    eval_strategy = "epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=CONFIG.batch_size,
    per_device_eval_batch_size=CONFIG.batch_size,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=CONFIG.max_epoch,
    warmup_steps=100,
    optim="adamw_bnb_8bit",
    prediction_loss_only=True,
    gradient_accumulation_steps = gradient_accumulation_steps,
    bf16=True,
    bf16_full_eval=True,
    group_by_length=True,
)

Once we have our model, we can define a Trainer by passing it all the objects constructed up to now — the model, the training_args, the training and validation datasets, the tokenizer and the data collator:

In [72]:
from transformers import Trainer

trainer = Trainer(
    lora_model,
    args,
    train_dataset=preprocessed_train_dataset,
    eval_dataset=preprocessed_dev_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


/tmp/ipykernel_5500/339621388.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


To fine-tune the model on our dataset, we just have to call the [train() function](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.Trainer.train) of our Trainer. However, the [wandb library](https://docs.wandb.ai/guides) is used and it requires to have a [wandb account and login](https://docs.wandb.ai/guides/integrations/huggingface/).

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
The model is already on multiple devices. Skipping the move to device specified in `args`.


Epoch,Training Loss,Validation Loss
1,1.321500,1.288383
2,1.240900,1.242298
3,1.198400,1.222983
4,1.169000,1.211728
5,1.147800,1.208933


TrainOutput(global_step=5160, training_loss=1.255916861600654, metrics={'train_runtime': 15620.4281, 'train_samples_per_second': 21.139, 'train_steps_per_second': 0.33, 'total_flos': 7.339969662025728e+17, 'train_loss': 1.255916861600654, 'epoch': 5.0})

In [ ]:
# save the model
trainer.save_model(CONFIG.output_model_dir)

In [73]:
# Load the model
from peft import PeftModel
lora_model = PeftModel.from_pretrained(model, CONFIG.output_model_dir)

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


## Inference

At inference time, it is recommended to use [generate()](https://huggingface.co/docs/transformers/en/main_classes/text_generation#transformers.GenerationMixin.generate). This method takes care of encoding the input and auto-regressively generates the decoder output. Check out [this blog post](https://huggingface.co/blog/how-to-generate) to know all the details about generating text with Transformers.

Let us first load the default inference parameters of Llama-2: 

In [74]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    checkpoint,
)

print(generation_config)

GenerationConfig {
  "bos_token_id": 1,
  "do_sample": true,
  "eos_token_id": 2,
  "max_length": 4096,
  "pad_token_id": 0,
  "temperature": 0.6,
  "top_p": 0.9
}



As observed, the default search strategy for Llama-2 is Top-p with probability 0.9 and temperature 0.6 ($0<T<1$ amplifies output probability differences and makes output more deterministic). [The search strategy can be selected](https://huggingface.co/docs/transformers/en/generation_strategies) at inference time. 

First, the test set is divided in small batches to reduce GPU memory comsumption:

In [75]:
batch_tokenized_test = preprocessed_test_dataset.batch(CONFIG.batch_size)

Batching examples:   0%|          | 0/14342 [00:00<?, ? examples/s]

Batching examples: 100%|██████████| 14342/14342 [00:00<00:00, 20845.62 examples/s]


Batches are provided to the [generate()](https://huggingface.co/docs/transformers/en/main_classes/text_generation#transformers.GenerationMixin.generate) together with inference parameters to define the search strategy. In this case, num_beams = 1 and do_sample = False means greedy search. 

In [77]:
import tqdm
number_of_batches = len(batch_tokenized_test["input_ids"])
output_sequences = []
all_sources = []
for i in tqdm.tqdm(range(number_of_batches)):
    all_sources.append(batch_tokenized_test["input_ids"][i])
    with torch.no_grad():
        output_batch = lora_model.generate(
            generation_config=generation_config, 
            input_ids=torch.tensor(batch_tokenized_test["input_ids"][i]).cuda(), 
            attention_mask=torch.tensor(batch_tokenized_test["attention_mask"][i]).cuda(), 
            max_length = max_tok_len, 
            num_beams=1, 
            do_sample=False,)
    output_sequences.extend(output_batch)

  0%|          | 0/1793 [00:00<?, ?it/s]

100%|██████████| 1793/1793 [36:07<00:00,  1.21s/it]


## Evaluation

The output of the model is automatically evaluated compared to the reference translations. To this purpose, we use the [Evaluate library](https://huggingface.co/docs/evaluate) which includes the definition of generic and task-specific metrics. In our case, we use the [BLEU metric](https://huggingface.co/spaces/evaluate-metric/bleu), or to be more precise, [sacreBLEU](https://huggingface.co/spaces/evaluate-metric/sacrebleu).

In [78]:
from evaluate import load

metric_bleu = load("sacrebleu")
metric_comet = load("comet")

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 9589.17it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.6. To apply the upgrade to your files permanently, run `py

The example below performs a basic post-processing to decode the predictions and extract the translation:

In [79]:
import re

def compute_metrics(sample, output_sequences):
    inputs = [f"{task_prefix}{src}: {s} = {tgt}: "  for s in sample["source_text"]]
    preds = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)
    # print(inputs)
    # print(preds)
    for i, (input,pred) in enumerate(zip(inputs,preds)):
      pred = re.search(r'^.*\n',pred.removeprefix(input).lstrip())
      if pred is not None:
        preds[i] = pred.group()[:-1]
      else:
        preds[i] = ""
    # print(sample["source_text"])
    # print(sample["dest_text"])
    # print(preds)
    result_bleu = metric_bleu.compute(
       predictions=preds, 
       references=sample["dest_text"]
    )
    result_comet = metric_comet.compute(
        sources=sample["source_text"],
        predictions=preds, 
        references=sample["dest_text"]
    )
    result = {
      "bleu": result_bleu["score"],
      "comet": result_comet["mean_score"]
      }
    return result

In [80]:
result = compute_metrics(preprocessed_test_dataset,output_sequences)
print(f'BLEU score: {result["bleu"]:0.4f}')
print(f'COMET score: {result["comet"]:0.4f}')

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return _C._get_float32_matmul_precision()
You a

BLEU score: 22.7613
COMET score: 0.7910


In [81]:
from maikol_utils.print_utils import print_separator

# all_sources already contains raw text strings
# output_sequences contains token IDs that need to be decoded
decoded_outputs = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)

# Get reference translations from test set
test_references = tokenized_datasets["test"]["dest_text"]

# Print first 10 examples
for i, (source, output, reference) in enumerate(zip(all_sources[:10], decoded_outputs[:10], test_references[:10])):
    print_separator(f"Example {i+1}:")
    print(f"Source:      {source}")
    print(f"Translation: {output}")
    print(f"Reference:   {reference}")

________________________________________________________________
                           Example 1:                           

Source:      [[2, 2, 2, 1, 4103, 9632, 515, 831, 304, 321, 29877, 29901, 13, 267, 29901, 633, 307, 1232, 288, 14736, 29892, 553, 29880, 398, 1182, 912, 29889, 353, 321, 29877, 29901, 29871], [2, 2, 2, 2, 2, 2, 1, 4103, 9632, 515, 831, 304, 321, 29877, 29901, 13, 267, 29901, 21135, 5821, 381, 1715, 425, 907, 655, 1290, 29889, 353, 321, 29877, 29901, 29871], [2, 2, 2, 1, 4103, 9632, 515, 831, 304, 321, 29877, 29901, 13, 267, 29901, 560, 3348, 11088, 316, 12374, 1647, 831, 19615, 417, 1702, 26161, 29889, 353, 321, 29877, 29901, 29871], [2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 4103, 9632, 515, 831, 304, 321, 29877, 29901, 13, 267, 29901, 5178, 14736, 316, 298, 29894, 562, 353, 321, 29877, 29901, 29871], [2, 2, 2, 2, 2, 2, 1, 4103, 9632, 515, 831, 304, 321, 29877, 29901, 13, 267, 29901, 2532, 29878, 8202, 767, 3090, 5291, 282, 709, 29889, 353, 321, 29877, 29901, 29871], [